# Suite CCONF — the C corpus

The conformance files this implementation writes for C (`python -m mbse.Programs.Conformance.write`), as for
TypeScript (TS5CONF): the source's frame, the `References.OfFrame` of the transaction its parse gives, and the text its
translation unit prints, for `c23.c`. The whole tree is values: the frame binds only the workspace (`ws`) and the unit
(`unit`), and its one assignment gives the unit its `body`, every declaration and statement nested in it, the
statements, members and groups within terms among them, as one value. That the files are current, P3CONF-01 checks,
the writer writing every language ported so far.

In [ ]:
import json

from mbse.Programs import C
from mbse.Programs.Conformance import write
from mbse.Schemas.Framework import Comparison, Plain

CORPUS = write.DEFAULT
SOURCES = write.ROOT / "sources"
NAME = "c23.c"

<a id="CCONF-01"></a>
## CCONF-01 · A frame read back compares equal, and is written again as it was

In [ ]:
text = (CORPUS / "c23.json").read_text(encoding="utf-8")
store = C.store()
schema = store.registered("References.OfFrame")
mine = json.loads(write.frame(C.store(), C.Parse.parse((SOURCES / NAME).read_text(encoding="utf-8"), NAME)))
read = Plain.FromPlain(store).Value(schema, json.loads(text))
again = Plain.FromPlain(store).Value(schema, mine)
assert Comparison.OfObject(schema, read).compare(Comparison.OfObject(schema, again)) == 0
assert json.dumps(Plain.ToPlain(store).Value(schema, read), indent=2, ensure_ascii=False) + "\n" == text
frame = json.loads(text)
assert set(frame) == {"binds", "assigns"} and set(frame["binds"]) == {"ws", "unit"}
(assign,) = frame["assigns"]  # the unit, given its whole tree as one nested value
assert assign["object"] == frame["binds"]["unit"] and set(assign["properties"]) == {"body"}

<a id="CCONF-02"></a>
## CCONF-02 · The printed corpus parses to the tree its source does, and prints as it is

No C compiler is at hand to compare meanings, as `ast` does for Python, so the printed text is parsed again: its tree
is the source's but for the lines its statements begin on, and it prints back as it is.

In [ ]:
def unlined(value):
    """A plain value without its `line`s: where each statement begins, which the printer's layout moves."""
    if isinstance(value, dict):
        return {key: unlined(item) for key, item in value.items() if key != "line"}
    return [unlined(item) for item in value] if isinstance(value, list) else value


source = (SOURCES / NAME).read_text(encoding="utf-8")
printed = (CORPUS / NAME).read_text(encoding="utf-8")
trees = [json.loads(write.frame(C.store(), C.Parse.parse(text, NAME)))["assigns"][0]["properties"]
         for text in (source, printed)]
assert unlined(trees[0]) == unlined(trees[1])
store = C.store()
store.transaction(C.Parse.parse(printed, NAME))
assert C.Print.print_module(store, NAME) == printed